# Mini Eval: Promptları Ölçerek Karşılaştırma

Bu notebook'un amacı, aynı modeli kullanıp **farklı prompt tasarımlarını küçük bir test seti üzerinde karşılaştırmaktır**.

Bu örnekte görevimiz: kısa e-postaları sınıflandırmak.

Her örnek için modelden şu JSON çıktısını bekliyoruz:

```json
{
  "priority": "high | medium | low",
  "category": "meeting | support | sales | other"
}
```

Ölçtüğümüz metrikler:
- **accuracy_priority**
- **accuracy_category**
- **exact_match**
- **valid_json_rate**
- **avg_latency_sec**
- **avg_output_tokens**
- **avg_total_tokens**

> Not: Bu notebook güncel OpenAI Python SDK'sı ve **Responses API** mantığıyla hazırlanmıştır.


## 1) Kurulum

Bu notebook'u çalıştırmadan önce:
1. OpenAI API anahtarınızı alın
2. Ortam değişkeni olarak ayarlayın:

**macOS / Linux**
```bash
export OPENAI_API_KEY="YOUR_KEY"
```

**Windows PowerShell**
```powershell
setx OPENAI_API_KEY "YOUR_KEY"
```

İsterseniz notebook içinde geçici olarak da tanımlayabilirsiniz, ama güvenlik açısından ortam değişkeni tercih edilir.


In [2]:
# Gerekirse açın:
!pip install -q openai pandas

import os
import json
import time
from typing import Dict, Any, List

import pandas as pd
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY ortam değişkeni bulunamadı.")

client = OpenAI(api_key=api_key)

# İsterseniz daha sonra değiştirebilirsiniz.
MODEL = "gpt-5-mini"
print("Model:", MODEL)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Model: gpt-5-mini


## 2) Küçük bir değerlendirme veri kümesi

Bu veri kümesi bilerek küçük tutuldu. Derste hızlı deney yapmak için yeterli.

İsterseniz:
- örnek sayısını artırabilir,
- kendi ders alanınıza uygun örnekler ekleyebilir,
- siber güvenlik ticket'ları / çağrı merkezi mesajları / öğrenci e-postaları gibi alanlara uyarlayabilirsiniz.


In [3]:

dataset = [
    {
        "id": 1,
        "email": "Our production API has been down since 09:10. Multiple customers are affected. Need urgent help.",
        "priority": "high",
        "category": "support",
    },
    {
        "id": 2,
        "email": "Can we move tomorrow's curriculum meeting from 10:00 to 11:30?",
        "priority": "medium",
        "category": "meeting",
    },
    {
        "id": 3,
        "email": "I would like a demo of your identity platform for our telecom team next week.",
        "priority": "medium",
        "category": "sales",
    },
    {
        "id": 4,
        "email": "Just sharing the slides from today's seminar. No action needed.",
        "priority": "low",
        "category": "other",
    },
    {
        "id": 5,
        "email": "Password reset link is not working and I cannot access the admin panel.",
        "priority": "high",
        "category": "support",
    },
    {
        "id": 6,
        "email": "Would Thursday afternoon work for a short project sync?",
        "priority": "medium",
        "category": "meeting",
    },
    {
        "id": 7,
        "email": "We are evaluating PAM solutions and would like pricing information for 500 users.",
        "priority": "medium",
        "category": "sales",
    },
    {
        "id": 8,
        "email": "Thanks for the update. Everything looks good on my side.",
        "priority": "low",
        "category": "other",
    },
    {
        "id": 9,
        "email": "Critical bug: users are being charged twice after checkout.",
        "priority": "high",
        "category": "support",
    },
    {
        "id": 10,
        "email": "Please find attached the revised budget sheet for your reference.",
        "priority": "low",
        "category": "other",
    },
]

df_gold = pd.DataFrame(dataset)
df_gold


,id,email,priority,category
0,1,Our production API has been down since 09:10. ...,high,support
1,2,Can we move tomorrow's curriculum meeting from...,medium,meeting
2,3,I would like a demo of your identity platform ...,medium,sales
3,4,Just sharing the slides from today's seminar. ...,low,other
4,5,Password reset link is not working and I canno...,high,support
5,6,Would Thursday afternoon work for a short proj...,medium,meeting
6,7,We are evaluating PAM solutions and would like...,medium,sales
7,8,Thanks for the update. Everything looks good o...,low,other
8,9,Critical bug: users are being charged twice af...,high,support
9,10,Please find attached the revised budget sheet ...,low,other


## 3) Karşılaştırılacak prompt adayları

Amaç: öğrenciler aynı görev için farklı prompt yazımlarının sonucu nasıl etkilediğini görsün.

Burada 3 prompt adayı var:
- **prompt_v1**: çok kısa
- **prompt_v2**: görev ve etiketler daha net
- **prompt_v3**: karar kuralları ve format beklentisi daha açık

Ders sırasında öğrencilere kendi promptlarını yazdırıp bunları da karşılaştırabilirsiniz.


In [4]:

PROMPTS = {
    "prompt_v1_minimal": """Classify the email and return JSON with priority and category.""",

    "prompt_v2_clearer": """You are an email triage assistant.
Classify the user's email into:
- priority: high, medium, or low
- category: meeting, support, sales, or other

Return JSON only with keys: priority, category.""",

    "prompt_v3_rule_based": """You are an email triage assistant.

Task:
Read the email and classify it using the labels below.

Priority labels:
- high: urgent issue, outage, broken access, business-critical problem, immediate action needed
- medium: normal request, scheduling, product inquiry, follow-up needed
- low: informational note, thanks, no clear action needed

Category labels:
- meeting: scheduling, rescheduling, calendar coordination
- support: bug, outage, access problem, technical help request
- sales: pricing, demo, purchase interest, vendor evaluation
- other: anything else

Rules:
- Return valid JSON only
- Do not include markdown
- Do not include explanations
- Use exactly these keys: priority, category
- Use only allowed label values

Expected format:
{"priority": "medium", "category": "meeting"}"""
}

list(PROMPTS.keys())


['prompt_v1_minimal', 'prompt_v2_clearer', 'prompt_v3_rule_based']

## 4) API çağrısı yardımcı fonksiyonları

Bu bölüm:
- modele istek gönderir,
- metni alır,
- JSON parse etmeye çalışır,
- token kullanımını ve süreyi kaydeder.


In [5]:

ALLOWED_PRIORITIES = {"high", "medium", "low"}
ALLOWED_CATEGORIES = {"meeting", "support", "sales", "other"}


def extract_text_from_response(response) -> str:
    """
    Responses API nesnesinden metni mümkün olduğunca reliable biçimde çıkarmaya çalışır.
    """
    # Yeni SDK'larda çoğu zaman bu alan vardır.
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text.strip()

    # Fallback: output listesi üzerinden gez
    try:
        parts = []
        for item in response.output:
            content = getattr(item, "content", None)
            if content:
                for c in content:
                    text = getattr(c, "text", None)
                    if text:
                        parts.append(text)
        if parts:
            return "\\n".join(parts).strip()
    except Exception:
        pass

    # En son çare
    return ""


def call_model(email_text: str, prompt_text: str, model: str = MODEL) -> Dict[str, Any]:
    start = time.perf_counter()

    response = client.responses.create(
        model=model,
        instructions=prompt_text,
        input=f"Email:\\n{email_text}"
    )

    latency = time.perf_counter() - start
    raw_text = extract_text_from_response(response)

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) if usage else None
    output_tokens = getattr(usage, "output_tokens", None) if usage else None
    total_tokens = getattr(usage, "total_tokens", None) if usage else None

    parsed = None
    valid_json = False
    parse_error = None

    try:
        parsed = json.loads(raw_text)
        valid_json = isinstance(parsed, dict)
    except Exception as e:
        parse_error = str(e)

    return {
        "raw_text": raw_text,
        "parsed": parsed,
        "valid_json": valid_json,
        "parse_error": parse_error,
        "latency_sec": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }


def score_prediction(gold_priority: str, gold_category: str, parsed: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(parsed, dict):
        return {
            "pred_priority": None,
            "pred_category": None,
            "priority_correct": 0,
            "category_correct": 0,
            "exact_match": 0,
            "label_space_valid": 0,
        }

    pred_priority = parsed.get("priority")
    pred_category = parsed.get("category")

    label_space_valid = int(
        pred_priority in ALLOWED_PRIORITIES and pred_category in ALLOWED_CATEGORIES
    )

    priority_correct = int(pred_priority == gold_priority)
    category_correct = int(pred_category == gold_category)
    exact_match = int(priority_correct == 1 and category_correct == 1)

    return {
        "pred_priority": pred_priority,
        "pred_category": pred_category,
        "priority_correct": priority_correct,
        "category_correct": category_correct,
        "exact_match": exact_match,
        "label_space_valid": label_space_valid,
    }


## 5) Tüm promptları tüm örnekler üzerinde çalıştır

Bu hücre her promptu veri kümesindeki her örneğe uygular ve detaylı sonuçları toplar.

> Not: API çağrısı yaptığı için bir miktar ücret ve süre oluşur.


In [6]:

all_rows = []

for prompt_name, prompt_text in PROMPTS.items():
    print(f"Running: {prompt_name}")
    for item in dataset:
        result = call_model(item["email"], prompt_text, model=MODEL)
        scores = score_prediction(item["priority"], item["category"], result["parsed"])

        row = {
            "prompt_name": prompt_name,
            "id": item["id"],
            "email": item["email"],
            "gold_priority": item["priority"],
            "gold_category": item["category"],
            **result,
            **scores,
        }
        all_rows.append(row)

df_results = pd.DataFrame(all_rows)
df_results.head()


Running: prompt_v1_minimal
Running: prompt_v2_clearer
Running: prompt_v3_rule_based


,prompt_name,id,email,gold_priority,gold_category,raw_text,parsed,valid_json,parse_error,latency_sec,input_tokens,output_tokens,total_tokens,pred_priority,pred_category,priority_correct,category_correct,exact_match,label_space_valid
0,prompt_v1_minimal,1,Our production API has been down since 09:10. ...,high,support,"{\n ""priority"": ""critical"",\n ""category"": ""p...","{'priority': 'critical', 'category': 'producti...",True,None,9.071504,46,414,460,critical,production_api_outage,0,0,0,0
1,prompt_v1_minimal,2,Can we move tomorrow's curriculum meeting from...,medium,meeting,"{""priority"":""normal"",""category"":""scheduling""}","{'priority': 'normal', 'category': 'scheduling'}",True,None,5.946720,43,228,271,normal,scheduling,0,0,0,0
2,prompt_v1_minimal,3,I would like a demo of your identity platform ...,medium,sales,"{""priority"":""high"",""category"":""demo_request""}","{'priority': 'high', 'category': 'demo_request'}",True,None,6.012981,41,281,322,high,demo_request,0,0,0,0
3,prompt_v1_minimal,4,Just sharing the slides from today's seminar. ...,low,other,"{""priority"":""low"",""category"":""Meeting material...","{'priority': 'low', 'category': 'Meeting mater...",True,None,5.432794,37,288,325,low,Meeting materials (FYI),1,0,0,0
4,prompt_v1_minimal,5,Password reset link is not working and I canno...,high,support,"{""priority"":""high"",""category"":""password_reset""}","{'priority': 'high', 'category': 'password_res...",True,None,4.803780,39,270,309,high,password_reset,1,0,0,0


## 6) Özet metrik tablosu

Burada prompt bazında özet metrikleri görüyoruz.


In [7]:

summary = (
    df_results.groupby("prompt_name", as_index=False)
    .agg(
        accuracy_priority=("priority_correct", "mean"),
        accuracy_category=("category_correct", "mean"),
        exact_match=("exact_match", "mean"),
        valid_json_rate=("valid_json", "mean"),
        label_space_valid_rate=("label_space_valid", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
    )
    .sort_values(by=["exact_match", "valid_json_rate", "avg_total_tokens"], ascending=[False, False, True])
)

summary


,prompt_name,accuracy_priority,accuracy_category,exact_match,valid_json_rate,label_space_valid_rate,avg_latency_sec,avg_output_tokens,avg_total_tokens
2,prompt_v3_rule_based,1.0,1.0,1.0,1.0,1.0,2.600694,96.8,293.8
1,prompt_v2_clearer,0.8,0.9,0.7,1.0,1.0,2.777235,129.1,203.1
0,prompt_v1_minimal,0.3,0.0,0.0,1.0,0.0,5.889303,287.2,326.2


## 7) Hata analizi

Sadece özet tabloya bakmak yetmez. Hangi örneklerde hata yapıldığını görmek gerekir.

Bu hücre exact match başarısız olan örnekleri gösterir.


In [8]:

errors = df_results[df_results["exact_match"] == 0].copy()

cols = [
    "prompt_name",
    "id",
    "email",
    "gold_priority",
    "gold_category",
    "pred_priority",
    "pred_category",
    "valid_json",
    "raw_text",
]

errors[cols].sort_values(["prompt_name", "id"])


,prompt_name,id,email,gold_priority,gold_category,pred_priority,pred_category,valid_json,raw_text
0,prompt_v1_minimal,1,Our production API has been down since 09:10. ...,high,support,critical,production_api_outage,True,"{\n ""priority"": ""critical"",\n ""category"": ""p..."
1,prompt_v1_minimal,2,Can we move tomorrow's curriculum meeting from...,medium,meeting,normal,scheduling,True,"{""priority"":""normal"",""category"":""scheduling""}"
2,prompt_v1_minimal,3,I would like a demo of your identity platform ...,medium,sales,high,demo_request,True,"{""priority"":""high"",""category"":""demo_request""}"
3,prompt_v1_minimal,4,Just sharing the slides from today's seminar. ...,low,other,low,Meeting materials (FYI),True,"{""priority"":""low"",""category"":""Meeting material..."
4,prompt_v1_minimal,5,Password reset link is not working and I canno...,high,support,high,password_reset,True,"{""priority"":""high"",""category"":""password_reset""}"
5,prompt_v1_minimal,6,Would Thursday afternoon work for a short proj...,medium,meeting,Normal,Meeting request,True,"{""priority"":""Normal"",""category"":""Meeting reque..."
6,prompt_v1_minimal,7,We are evaluating PAM solutions and would like...,medium,sales,High,"Sales Inquiry - Pricing Request (PAM, 500 users)",True,"{""priority"":""High"",""category"":""Sales Inquiry -..."
7,prompt_v1_minimal,8,Thanks for the update. Everything looks good o...,low,other,low,acknowledgement,True,"{""priority"":""low"",""category"":""acknowledgement""}"
8,prompt_v1_minimal,9,Critical bug: users are being charged twice af...,high,support,critical,billing_bug,True,"{""priority"":""critical"",""category"":""billing_bug""}"
9,prompt_v1_minimal,10,Please find attached the revised budget sheet ...,low,other,Normal,Budget,True,"{\n ""priority"": ""Normal"",\n ""category"": ""Bud..."


## 8) Basit bir maliyet/performans yorumu

Bu bölüm otomatik yorum üretmez; öğrencilerin kendisinin yorumlaması için bırakılmıştır.

Sorulabilecek sorular:
1. En yüksek **exact_match** hangi promptta?
2. En düşük token maliyeti hangisinde?
3. Daha uzun prompt her zaman daha mı iyi?
4. JSON uyumu ile doğruluk birlikte mi artıyor?
5. Ders alanınıza göre hangi metriği daha önemli kabul etmelisiniz?

Örnek yorum şablonu:
- `prompt_vX`, doğruluk açısından en iyi sonucu verdi.
- Ancak token maliyeti daha yüksekti.
- `prompt_vY` daha ucuzdu ama bazı örneklerde kategori karışıklığı yaptı.
- Dolayısıyla seçim yalnızca kaliteye değil, maliyet ve format güvenilirliğine göre yapılmalıdır.


## 9) İsteğe bağlı geliştirmeler

Bu notebook'u derste büyütmek için:
- veri kümesini 50+ örneğe çıkarın
- train/dev/test ayrımı yapın
- öğrencilerin kendi promptlarını ekleyin
- structured outputs kullanarak `valid_json_rate` sorununu azaltın
- farklı modelleri de karşılaştırın
- aynı promptu birden fazla kez çalıştırıp kararlılığı ölçün
- siber güvenlik biletleri, SOC alarm özetleri veya öğrenci e-postalarıyla alan uyarlaması yapın


## 10) İleri seviye ödev fikri

Öğrencilerden şunu isteyebilirsiniz:

> Aynı veri kümesinde üç prompt adayı tasarlayın.  
> Her prompt için doğruluk, JSON uyumu, latency ve token kullanımını karşılaştırın.  
> Ardından en iyi promptu yalnızca sezgisel olarak değil, metriklerle savunun.

Bu, prompt engineering'in “güzel yazma” değil, **ölçülebilir optimizasyon** işi olduğunu çok net gösterir.
